<a href="https://colab.research.google.com/github/Andreas-Lukito/Stock_Sentiment_Analysis/blob/dev%2Fandreas/notebooks/01_text_cleaning_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# News Data Cleaning

## Import Libraries

In [13]:
!pip install python-dotenv cloudscraper newspaper3k tqdm contractions emoji lxml_html_clean

In [14]:
# Common Libraries
import numpy as np
import pandas as pd
import os
import sys

# Cleaner output
from tqdm import tqdm
from IPython.display import clear_output

# Google Colab Setup
from google.colab import drive
drive.mount('/content/drive')
project_path = "/content/drive/MyDrive/stock_news_sentiment_analysis"

## Add the path to text preprocessor
sys.path.append(os.path.abspath(os.path.join(project_path, "lib")))

# Text preprocessing
from preprocessor import clean_text
from scraper import extract_text_from_url, scrape_dataframe

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import the dataset

In [15]:
news_data = pd.read_csv(os.path.join(project_path, "news_cache/catgorized_data/categorized_news_data2.csv"), sep=",")

##

In [16]:
news_data.head()

,index,uuid,title,description,keywords,snippet,url,image_url,language,published_at,source,relevance_score,entities,similar,sentiment,text,clean_text,categorical_sentiment_3_class,length
0,0,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
1,1,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259
2,2,9084e5f1-75f5-4f15-aa3d-0676073b4aaf,Global week ahead: The start of a Santa Rally ...,NaN,"STOXX 600, business news",And just like that... December is upon us. It'...,https://www.cnbc.com/2025/11/30/global-week-ah...,https://image.cnbcfm.com/api/v1/image/10823257...,en,2025-11-30T05:10:58.000000Z,cnbc.com,NaN,"[{'symbol': 'M', 'name': ""Macy's, Inc."", 'exch...",[],0.6908,And just like that... December is upon us. It'...,and just like that december is upon us it is b...,positive,493
3,3,487e6a88-d3c2-4ae1-8dc2-26af6b31d688,2025: The Year Of Alphabet (GOOG),No stock has seen a bigger jump recently than ...,NaN,vzphotos/iStock Editorial via Getty Images\n\n...,https://seekingalpha.com/article/4848680-2025-...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:30:00.000000Z,seekingalpha.com,NaN,"[{'symbol': 'GOOGL', 'name': 'Alphabet Inc.', ...",[],0.0000,vzphotos/iStock Editorial via Getty Images\n\n...,vzphotos istock editorial via getty images sin...,neutral,42
4,4,92b5c2bd-d324-4ae8-b115-2cfd95a8fa98,Why I'm Doubling Down On My Adobe Position (NA...,"Adobe's revenue is highly predictable, driven ...",NaN,To say that Adobe ( ADBE ) stock has not had a...,https://seekingalpha.com/article/4848762-why-i...,https://static.seekingalpha.com/cdn/s3/uploads...,en,2025-11-30T05:25:01.000000Z,seekingalpha.com,NaN,"[{'symbol': 'ADBE', 'name': 'Adobe Inc.', 'exc...",[],0.0000,To say that Adobe ( ADBE ) stock has not had a...,to say that adobe adbe stock has not had a goo...,neutral,259


## Imputing the `Text`

Since the previous method of fixing the contents of the article iesults in a more worse model, we will refetch the articles

In [ ]:
raw_rescraped_output_path = os.path.join(project_path, "news_cache/catgorized_data/categorized_news_data_rescraped_raw.csv")

news_data = scrape_dataframe(
    news_data,
    url_col="url",
    text_col="text",
    max_workers=5,
    batch_size=200,
    output_path=raw_rescraped_output_path,
)


Batch 21/386 (200 rows)


Batch 21:   0%|          | 0/200 [00:00<?, ?it/s]

Attempt 1 failed: 403 Forbidden - bot detection triggered. Retrying in 5s…


Batch 21:   0%|          | 1/200 [00:07<24:02,  7.25s/it]

✓ https://www.zacks.com/stock/news/2779638/chain-bridge-bancorp-inc-cbna-q3-earnings-top-estimates?cid=CS-ZC-FT-tale_of_the_tape|yseop_template_4-2779638


...

Attempt 1 failed: 403 Forbidden - bot detection triggered. Retrying in 5s…
✓ https://stockmarketwatch.com/stock-market-news/us-stocks-climb-on-tech-and-ai-hype-visa-edison-international-beat-estimates-while-mondelez-ea-face-headwinds/55225/


Batch 21:  66%|██████▌   | 132/200 [09:47<03:35,  3.16s/it]

Failed (All 3 attempts failed for: https://www.manilatimes.net/2025/10/29/tmt-newswire/globenewswire/orasure-technologies-appoints-steven-k-boyd-to-its-board-of-directors/2210612
Last error: 403 Forbidden - bot detection triggered), using fallback | https://www.manilatimes.net/2025/10/29/tmt-newswire/globenewswire/orasure-technologies-appoints-steven-k-boyd-to-its-board-of-directors/2210612
Attempt 1 failed: 403 Forbidden - bot detection triggered. Retrying in 5s…
Attempt 2 failed: 403 Forbidden - bot detection triggered. Retrying in 10s…
Attempt 2 failed: 403 Forbidden - bot detection triggered. Retrying in 10s…
Attempt 1 failed: 403 Forbidden - bot detection triggered. Retrying in 5s…
Attempt 2 failed: 403 Forbidden - bot detection triggered. Retrying in 10s…


Batch 21:  66%|██████▋   | 133/200 [10:07<09:09,  8.21s/it]

Failed (All 3 attempts failed for: https://www.manilatimes.net/2025/10/29/tmt-newswire/globenewswire/first-community-bankshares-inc-announces-third-quarter-2025-results-and-quarterly-cash-dividend/2210611
Last error: 403 Forbidden - bot detection triggered), using fallback | https://www.manilatimes.net/2025/10/29/tmt-newswire/globenewswire/first-community-bankshares-inc-announces-third-quarter-2025-results-and-quarterly-cash-dividend/2210611
Attempt 2 failed: 403 Forbidden - bot detection triggered. Retrying in 10s…


## Text Cleaning

## Text Cleaning

In [ ]:

categorized_data_path = os.path.join(project_path,f"news_cache/catgorized_data/categorized_news_data_rescraped.csv")
categorized_data_path_folder = os.path.join(project_path,f"news_cache/catgorized_data/")
os.makedirs(categorized_data_path_folder, exist_ok=True)
overwrite_clean_data = True


In [ ]:
# tqdm for cleaner output
tqdm.pandas(desc="Cleaning the Text", unit="news")

# We will cache the data so that it will load faster
if os.path.exists(categorized_data_path) and not overwrite_clean_data:
    print("Loading cached dataset...")
    news_data = pd.read_csv(categorized_data_path)
    print("Cached dataset loaded")

elif os.path.exists(categorized_data_path) and overwrite_clean_data:
    print("Overwriting old data and caching new data...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False, # Since we will be using a transformer model, we will not remove stop words since it can be useful for the model to understand the context of the sentence.
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Done Overwriting old data and caching new data...")

else:
    print("Creating and caching dataset...")
    # Clean the data
    news_data["clean_text"] = news_data["text"].progress_apply(
                                                        lambda x: clean_text(
                                                            text = x,
                                                            tokenize=False,
                                                            remove_stop_words= False,
                                                            remove_emojis="keep"
                                                            )
                                                        )
    news_data.to_csv(categorized_data_path, index=False)
    print("Finished Caching")